In [1]:
!pip install -q transformers sentencepiece sentence-transformers language-tool-python nltk pandas textstat

In [2]:
import torch
import nltk
import pandas as pd
import language_tool_python

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

print("All libraries imported successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All libraries imported successfully!


In [3]:
model_name = "Vamsi/T5_Paraphrase_Paws"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)
model.eval()

similarity_model = SentenceTransformer("all-MiniLM-L6-v2")

print("T5 model loaded successfully!")
print("Device:", device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

... (11 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

T5 model loaded successfully!
Device: cuda


In [4]:
grammar_tool = language_tool_python.LanguageTool(
    'en-US',
    language_tool_download_version='6.5'
)

print("Grammar and spelling checker is ready!")

Grammar and spelling checker is ready!


In [5]:
from difflib import SequenceMatcher

def paraphrase_sentence(sentence):

    text = "paraphrase: " + sentence + " </s>"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            top_k=120,
            top_p=0.95,
            temperature=1.2,
            repetition_penalty=1.2,
            num_return_sequences=10
        )

    paraphrases = []

    for output in outputs:

        result = tokenizer.decode(
            output,
            skip_special_tokens=True
        ).strip()

        if result and result.lower() != sentence.strip().lower():
            if result not in paraphrases:
                paraphrases.append(result)

    if not paraphrases:
        return sentence

    # Check meaning similarity
    original_vector = similarity_model.encode(
        sentence,
        convert_to_tensor=True
    )

    paraphrase_vectors = similarity_model.encode(
        paraphrases,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        original_vector,
        paraphrase_vectors
    )[0]

    best_sentence = sentence
    best_score = -1

    for i, candidate in enumerate(paraphrases):

        meaning = scores[i].item()

        text_similarity = SequenceMatcher(
            None,
            sentence.lower(),
            candidate.lower()
        ).ratio()

        difference = 1 - text_similarity

        # Select a sentence with similar meaning and different wording
        if meaning >= 0.90 and difference >= 0.08:

            final_score = meaning + (difference * 0.25)

            if final_score > best_score:
                best_score = final_score
                best_sentence = candidate

    return best_sentence

In [26]:
def paraphrase_text(text):

    if not text.strip():
        return "Please enter some text."

    # Avoid splitting B.Tech into two sentences
    text = text.replace("B.Tech", "BTech")

    sentences = sent_tokenize(text)

    paraphrased_sentences = []

    # Important words that should not be changed
    important_terms = [
        "nikhil krishna",
        "btech",
        "cr3",
        "7 years",
        "chemical cleaning",
        "oil flushing",
        "hydrocarbon decontamination",
        "refinery shutdown"
    ]

    for sentence in sentences:

        # Keep sentences with important details unchanged
        if any(term in sentence.lower() for term in important_terms):
            new_sentence = sentence

        else:
            new_sentence = paraphrase_sentence(sentence)

        paraphrased_sentences.append(new_sentence)

    # Join sentences
    final_text = " ".join(paraphrased_sentences)

    # Grammar correction
    final_text = grammar_tool.correct(final_text)

    # Restore the correct degree name
    final_text = final_text.replace("BTech", "B.Tech")
    final_text = final_text.replace("B. Tech", "B.Tech")

    return final_text

In [27]:
original_text = "Artificial intelligence is changing the way people work and learn."

paraphrased_text = paraphrase_text(original_text)

print("Original Text:")
print(original_text)

print("\nParaphrased Text:")
print(paraphrased_text)

Original Text:
Artificial intelligence is changing the way people work and learn.

Paraphrased Text:
Artificial intelligence changes the way humans work and learn.


In [28]:
def meaning_similarity(original, paraphrased):

    sentences = [original, paraphrased]

    embeddings = similarity_model.encode(
        sentences,
        convert_to_tensor=True
    )

    similarity = util.cos_sim(
        embeddings[0],
        embeddings[1]
    )

    score = similarity[0][0].item() * 100

    return round(score, 2)


similarity_score = meaning_similarity(
    original_text,
    paraphrased_text
)

print("Meaning Similarity:", similarity_score, "%")

Meaning Similarity: 92.75 %


In [29]:
from difflib import SequenceMatcher
import textstat

def check_quality(original, paraphrased):

    # Calculate how much the wording changed
    similarity = SequenceMatcher(
        None,
        original.lower(),
        paraphrased.lower()
    ).ratio()

    originality = (1 - similarity) * 100

    # Check grammar and spelling mistakes
    grammar_before = len(grammar_tool.check(original))
    grammar_after = len(grammar_tool.check(paraphrased))

    # Calculate readability
    readability = textstat.flesch_reading_ease(paraphrased)

    return {
        "Meaning Similarity (%)": meaning_similarity(original, paraphrased),
        "Wording Difference (%)": round(originality, 2),
        "Grammar Errors Before": grammar_before,
        "Grammar Errors After": grammar_after,
        "Readability Score": round(readability, 2)
    }


quality = check_quality(original_text, paraphrased_text)

for key, value in quality.items():
    print(key, ":", value)

Meaning Similarity (%) : 92.75
Wording Difference (%) : 15.62
Grammar Errors Before : 0
Grammar Errors After : 0
Readability Score : 37.9


In [35]:
test_sentences = [
    "Artificial intelligence is changing the way people work.",

    "Students can improve their knowledge by reading books regularly.",

    "The internet has made communication easier and faster.",

    "Machine learning helps computers identify patterns in data.",

    "The students is very happy because they has completed their assignment."
]

results = []

for i, sentence in enumerate(test_sentences):

    paraphrased = paraphrase_text(sentence)

    quality = check_quality(sentence, paraphrased)

    results.append({
        "Original Text": sentence,
        "Paraphrased Text": paraphrased,
        **quality
    })

results_df = pd.DataFrame(results)

pd.set_option("display.max_colwidth", None)

display(results_df)

,Original Text,Paraphrased Text,Meaning Similarity (%),Wording Difference (%),Grammar Errors Before,Grammar Errors After,Readability Score
0,Artificial intelligence is changing the way people work.,Artificial intelligence is changing the way people work.,100.00,0.00,0,0,29.52
1,Students can improve their knowledge by reading books regularly.,Students can improve their knowledge by regularly reading books.,99.73,15.62,0,0,47.30
2,The internet has made communication easier and faster.,The internet made communication easier and quicker.,91.89,12.38,0,0,6.36
3,Machine learning helps computers identify patterns in data.,Machine learning helps computers to elucidate patterns in data.,97.10,11.48,0,0,28.50
4,The students is very happy because they has completed their assignment.,The students are very happy because they have finished their assignment.,98.53,14.69,2,0,57.23


In [36]:
average_similarity = results_df["Meaning Similarity (%)"].mean()

average_originality = results_df["Wording Difference (%)"].mean()

average_readability = results_df["Readability Score"].mean()

grammar_before = results_df["Grammar Errors Before"].sum()

grammar_after = results_df["Grammar Errors After"].sum()

pass_rate = (
    results_df["Meaning Similarity (%)"] >= 75
).mean() * 100

print("AI PARAPHRASING TOOL - EVALUATION REPORT")
print("----------------------------------------")

print("Total Samples Tested:", len(results_df))

print("Average Meaning Similarity:",
      round(average_similarity, 2), "%")

print("Meaning Preservation Pass Rate:",
      round(pass_rate, 2), "%")

print("Average Wording Difference:",
      round(average_originality, 2), "%")

print("Average Readability Score:",
      round(average_readability, 2))

print("Grammar Errors Before:", grammar_before)

print("Grammar Errors After:", grammar_after)

AI PARAPHRASING TOOL - EVALUATION REPORT
----------------------------------------
Total Samples Tested: 5
Average Meaning Similarity: 97.45 %
Meaning Preservation Pass Rate: 100.0 %
Average Wording Difference: 10.83 %
Average Readability Score: 33.78
Grammar Errors Before: 2
Grammar Errors After: 0


In [37]:
print("AI-Powered Paraphrasing Tool")
print("---------------------------")

user_input = input("Enter your text: ")

if user_input.strip():

    result = paraphrase_text(user_input)

    print("\nOriginal Text:")
    print(user_input)

    print("\nParaphrased Text:")
    print(result)

    print("\nEvaluation Results:")

    evaluation = check_quality(user_input, result)

    for key, value in evaluation.items():
        print(key, ":", value)

else:
    print("Please enter valid text.")


AI-Powered Paraphrasing Tool
---------------------------
Enter your text: My name is Nikhil Krishna M S, and I'm from Trivandrum, Kerala. I have a B.Tech degree in Petroleum Engineering, and I'm currently working as a Chemical Engineer at CR3. I have around 7 years of experience in the oil and gas industry, mainly in Saudi Arabia. My work involves industrial chemical cleaning, oil flushing, hydrocarbon decontamination, and refinery shutdown activities

Original Text:
My name is Nikhil Krishna M S, and I'm from Trivandrum, Kerala. I have a B.Tech degree in Petroleum Engineering, and I'm currently working as a Chemical Engineer at CR3. I have around 7 years of experience in the oil and gas industry, mainly in Saudi Arabia. My work involves industrial chemical cleaning, oil flushing, hydrocarbon decontamination, and refinery shutdown activities

Paraphrased Text:
My name is Nikhil Krishna M S, and I'm from Trivandrum, Kerala. I have a Tech degree in Petroleum Engineering, and I'm currentl

In [38]:
results_df.to_csv(
    "paraphrasing_results.csv",
    index=False
)

# Create evaluation report
report = f"""
AI-POWERED PARAPHRASING TOOL
EVALUATION REPORT

Model: T5 Transformer
Total Samples: {len(results_df)}

Average Meaning Similarity: {average_similarity:.2f}%
Meaning Preservation Pass Rate: {pass_rate:.2f}%
Average Wording Difference: {average_originality:.2f}%
Average Readability Score: {average_readability:.2f}

Grammar Errors Before: {grammar_before}
Grammar Errors After: {grammar_after}

Conclusion:
The tool uses a pretrained T5 transformer to generate
paraphrased sentences. Grammar and spelling are checked
using LanguageTool.

Semantic similarity is used to estimate whether the
original meaning is preserved. Wording difference is
used to measure how much the text has changed.

The model may sometimes change the meaning or produce
sentences that need further improvement. Therefore,
the final output should also be checked manually.
"""

with open("evaluation_report.txt", "w") as file:
    file.write(report)

print("Results saved successfully!")
print("Files created:")
print("1. paraphrasing_results.csv")
print("2. evaluation_report.txt")

Results saved successfully!
Files created:
1. paraphrasing_results.csv
2. evaluation_report.txt
